# 05 - Federated Learning
Notebook ini menjalankan simulasi Flower (FedAvg) end-to-end menggunakan data hasil preprocessing.

In [1]:
from pathlib import Path
import json
import sys
from typing import List

import numpy as np
import torch
from sklearn.model_selection import train_test_split

try:
    import flwr as fl
except ImportError as exc:
    raise ImportError(
        "Package 'flwr' belum terpasang di environment aktif. Install dulu: pip install flwr"
    ) from exc


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate
    return cwd


PROJECT_ROOT = resolve_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from models import create_baseline_model


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

artifact = PROJECT_ROOT / 'data' / 'processed' / 'sequence_bundle.npz'
if not artifact.exists():
    raise FileNotFoundError('sequence_bundle.npz not found. Run 02_preprocessing.ipynb first.')

data = np.load(artifact, allow_pickle=True)
X = data['features'].astype(np.float32)
y_raw = data['labels']

labels_unique = sorted(set(y_raw.tolist()))
label_to_idx = {label: i for i, label in enumerate(labels_unique)}
y = np.array([label_to_idx[val] for val in y_raw], dtype=np.int64)

# Hold-out global test set for centralized evaluation after each round
X_train_full, X_test_global, y_train_full, y_test_global = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if len(labels_unique) > 1 else None,
)

num_clients = 5
rounds = 5
local_epochs = 1
batch_size = 64
learning_rate = 1e-3

rng = np.random.default_rng(42)
indices = np.arange(len(X_train_full))
rng.shuffle(indices)
client_indices = np.array_split(indices, num_clients)
client_sizes = [int(len(idx)) for idx in client_indices]

print('Total samples:', len(X))
print('Train samples (federated):', len(X_train_full))
print('Global test samples:', len(X_test_global))
print('Client sizes:', client_sizes)


def make_loader(x_arr: np.ndarray, y_arr: np.ndarray, shuffle: bool) -> torch.utils.data.DataLoader:
    ds = torch.utils.data.TensorDataset(
        torch.tensor(x_arr, dtype=torch.float32),
        torch.tensor(y_arr, dtype=torch.long),
    )
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


client_train_loaders = {
    cid: make_loader(X_train_full[idx], y_train_full[idx], shuffle=True)
    for cid, idx in enumerate(client_indices)
}

# Small local evaluation split per client
client_eval_loaders = {}
for cid, idx in enumerate(client_indices):
    x_part = X_train_full[idx]
    y_part = y_train_full[idx]
    if len(x_part) > 8:
        x_tr, x_ev, y_tr, y_ev = train_test_split(x_part, y_part, test_size=0.2, random_state=42)
        client_train_loaders[cid] = make_loader(x_tr, y_tr, shuffle=True)
        client_eval_loaders[cid] = make_loader(x_ev, y_ev, shuffle=False)
    else:
        client_eval_loaders[cid] = make_loader(x_part, y_part, shuffle=False)

global_test_loader = make_loader(X_test_global, y_test_global, shuffle=False)


def get_model_parameters(model: torch.nn.Module) -> List[np.ndarray]:
    return [param.detach().cpu().numpy() for _, param in model.state_dict().items()]


def set_model_parameters(model: torch.nn.Module, parameters: List[np.ndarray]) -> None:
    state_dict = model.state_dict()
    new_state_dict = {k: torch.tensor(v) for k, v in zip(state_dict.keys(), parameters)}
    model.load_state_dict(new_state_dict, strict=True)


def train_one_epoch(model: torch.nn.Module, loader: torch.utils.data.DataLoader) -> float:
    model.train()
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    total_loss = 0.0
    batches = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item())
        batches += 1

    return total_loss / max(batches, 1)


def evaluate_model(model: torch.nn.Module, loader: torch.utils.data.DataLoader) -> tuple[float, float]:
    model.eval()
    criterion = torch.nn.CrossEntropyLoss()
    total_loss = 0.0
    total = 0
    correct = 0
    batches = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += float(loss.item())
            pred = torch.argmax(logits, dim=1)
            correct += int((pred == yb).sum().item())
            total += int(yb.numel())
            batches += 1

    return total_loss / max(batches, 1), (correct / max(total, 1))


input_dim = int(X.shape[2])
num_classes = int(len(labels_unique))
global_model = create_baseline_model(
    input_dim=input_dim,
    hidden_dim=64,
    num_layers=2,
    num_classes=num_classes,
    device=device,
)

print(f'FL setup ready | input_dim={input_dim}, num_classes={num_classes}, rounds={rounds}')

Using device: cpu
Total samples: 14242
Train samples (federated): 11393
Global test samples: 2849
Client sizes: [2279, 2279, 2279, 2278, 2278]
FL setup ready | input_dim=3, num_classes=51, rounds=5


In [2]:
import sys
from pathlib import Path

# Add scripts dir to path to import run_fl_manual
PROJECT_ROOT = resolve_project_root()
scripts_dir = PROJECT_ROOT / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

import run_fl_manual
run_fl_manual.main()


Orchestrator: Starting FL Server Process...


Orchestrator: Starting FL Client Processes...


Orchestrator: Waiting for clients to complete...


Orchestrator: Waiting for server to conclude...
Orchestrator: Collecting round results...

Final global centralized evaluation: Loss=3.9376, Accuracy=0.0225
Saved federated learning metrics to C:\Users\anang\Downloads\Projek Keamanan Informasi\outputs\reports\fl_metrics.json
Saved global FL model to C:\Users\anang\Downloads\Projek Keamanan Informasi\outputs\models\fl_lstm.pt
